# 🤖 Agentic Coding Agent — Project 2 / 4 (Agentic AI Portfolio)

```
01 — Research Agent
02 — Coding Agent        ← this notebook
03 — Sales Agent
04 — Recruiting Agent
```

This notebook builds and runs an **autonomous coding agent** entirely inside
Google Colab, using the **Gemini free-tier API**. No OpenAI, no Docker, no
LangChain/LangGraph/CrewAI/AutoGen — the agent architecture (state, actions,
loop, tools) is implemented from first principles in plain Python.

**Architecture**

```
                  USER TASK
                      │
                      ▼
             ┌─────────────────┐
             │ REQUIREMENT     │
             │ ANALYZER        │
             └────────┬────────┘
                      ▼
             ┌─────────────────┐
             │    PLANNER      │
             └────────┬────────┘
                      ▼
             ┌─────────────────┐
             │ CODE GENERATOR  │
             └────────┬────────┘
                      ▼
             ┌─────────────────┐
             │ TEST GENERATOR  │
             └────────┬────────┘
                      ▼
             ┌─────────────────┐
             │ CODE EXECUTOR   │
             └────────┬────────┘
                      ▼
                 TEST RESULT
                 /        \
              PASS        FAIL
               │            │
               │            ▼
               │    ┌───────────────┐
               │    │ FAILURE       │
               │    │ ANALYZER      │
               │    └───────┬───────┘
               │            ▼
               │    ┌───────────────┐
               │    │ CODE FIXER    │
               │    └───────┬───────┘
               │            │
               │            └──────► RETEST
               │
               ▼
        FINAL IMPLEMENTATION
```

**How to use this notebook:** Run every cell top to bottom. Cells write the
project's `.py` files to the Colab filesystem using `Path.write_text(...)`,
then later cells import and run them — so what you see execute is exactly
what ships in the GitHub-ready `02-coding-agent/` folder.


In [1]:
# Install dependencies (Gemini SDK only — everything else is stdlib)
!pip install -q google-genai
print("\u2713 dependencies installed")

✓ dependencies installed


In [2]:
# Core imports used directly in this notebook
import os
import sys
import json
import shutil
from pathlib import Path

sys.path.insert(0, ".")  # so `import config`, `import llm`, etc. work
print("\u2713 imports ready")

✓ imports ready


In [3]:
# Load the Gemini API key from Colab Secrets — NEVER hard-code it.
#
# Setup (one time):
#   1. Click the \U0001F511 "Secrets" icon in the left sidebar of Colab.
#   2. Add a new secret named GEMINI_API_KEY with your key as the value.
#   3. Toggle "Notebook access" ON for this secret.
#   4. Get a free key at https://aistudio.google.com/apikey

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    # Fallback for non-Colab environments (e.g. local testing) — reads
    # from a real environment variable only, never a literal in code.
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    print("\u26a0 GEMINI_API_KEY not found. Add it to Colab Secrets before running the agent cells below.")
else:
    print("\u2713 Gemini API key loaded (not printed)")

✓ Gemini API key loaded (not printed)


In [4]:
# Cell 5 — Configuration
from pathlib import Path

Path("config.py").parent.mkdir(parents=True, exist_ok=True)
Path("config.py").write_text(
'"""\nconfig.py\n=========\nCentral configuration for the Agentic Coding Agent.\n\nAll tunables live here so the rest of the codebase never hard-codes\n"magic numbers" or model names. This keeps the project easy to adapt\nto whatever Gemini model the user\'s free-tier account has access to.\n"""\n\nfrom __future__ import annotations\n\nimport os\n\n# ---------------------------------------------------------------------------\n# Gemini model configuration\n# ---------------------------------------------------------------------------\n# The model name is intentionally kept configurable via an environment\n# variable. Free-tier Gemini access can vary by account/region, so the\n# architecture must not assume any single model name is always available.\n# Change this (or set the GEMINI_MODEL env var / Colab form field) to\n# whatever model your account currently has free-tier access to, e.g.\n# "gemini-2.0-flash", "gemini-2.5-flash", "gemini-1.5-flash", etc.\nGEMINI_MODEL: str = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")\n\n# ---------------------------------------------------------------------------\n# Agent loop bounds — mandatory to protect the free-tier quota\n# ---------------------------------------------------------------------------\n# 6 iterations is the minimum needed for the agent to actually use its\n# "self-fix" loop: PLAN -> GENERATE_CODE -> GENERATE_TESTS -> RUN_TESTS ->\n# ANALYZE_FAILURE+FIX -> RUN_TESTS. With fewer iterations the agent can\n# reach the failure state but never gets a turn to fix it — the diagram\n# in the README (FAILURE ANALYZER -> CODE FIXER) would never actually run.\nMAX_ITERATIONS: int = int(os.getenv("MAX_ITERATIONS", "6"))\n\n# ---------------------------------------------------------------------------\n# Safe execution settings\n# ---------------------------------------------------------------------------\nEXECUTION_TIMEOUT: int = int(os.getenv("EXECUTION_TIMEOUT", "10"))  # seconds\n\n# ---------------------------------------------------------------------------\n# Generation settings\n# ---------------------------------------------------------------------------\nGEMINI_TEMPERATURE_PLANNING: float = 0.2   # deterministic, structured output\nGEMINI_TEMPERATURE_CODEGEN: float = 0.3    # a little room for good code\nGEMINI_TEMPERATURE_FIX: float = 0.2        # focused, deterministic fixes\n\n# Max output tokens per call — keeps responses (and quota use) bounded.\n# Lowered from 2048 -> 1536: plenty for the small utility-style modules\n# this agent generates, while leaving more headroom under free-tier\n# tokens-per-minute limits.\nGEMINI_MAX_OUTPUT_TOKENS: int = int(os.getenv("GEMINI_MAX_OUTPUT_TOKENS", "1536"))\n\n# ---------------------------------------------------------------------------\n# Free-tier quota protection\n# ---------------------------------------------------------------------------\n# Free-tier Gemini keys are rate-limited (requests/minute) AND capped on\n# total requests (per day, or in your case as few as ~20 total). The three\n# settings below exist purely to keep this notebook well-behaved on a\n# constrained free key, so it produces clean, presentable output instead\n# of a raw 429 traceback halfway through a demo.\n\n# Minimum gap (seconds) enforced between consecutive *live* Gemini calls.\n# 6s -> at most 10 calls/minute, safely under typical free-tier RPM caps.\n# Raise this if your key\'s RPM limit is lower than 10.\nMIN_SECONDS_BETWEEN_CALLS: float = float(os.getenv("MIN_SECONDS_BETWEEN_CALLS", "6"))\n\n# Hard cap on how many *live* (non-cached) Gemini calls this notebook will\n# make in one run, across every cell. Once reached, the agent stops\n# gracefully (partial results are still shown) instead of burning through\n# the rest of a small quota and crashing on a 429. Tune this to match\n# whatever your key\'s actual limit is.\nGEMINI_SESSION_CALL_BUDGET: int = int(os.getenv("GEMINI_SESSION_CALL_BUDGET", "20"))\n\n# On-disk response cache so re-running a cell (e.g. while polishing this\n# notebook for GitHub) never re-spends quota on a prompt you already ran.\n# Identical (model, prompt, temperature, max_tokens) -> served from disk.\nGEMINI_CACHE_ENABLED: bool = os.getenv("GEMINI_CACHE_ENABLED", "1") != "0"\nGEMINI_CACHE_DIR: str = os.getenv("GEMINI_CACHE_DIR", ".gemini_cache")\n\n# ---------------------------------------------------------------------------\n# Misc\n# ---------------------------------------------------------------------------\nPROJECT_NAME: str = "Agentic Coding Agent"\nSUPPORTED_TASK_DOMAINS = (\n    "CLI applications",\n    "Data processing scripts",\n    "Algorithms",\n    "Utility functions",\n    "File processing",\n    "API client examples",\n    "Small automation scripts",\n    "Basic ML preprocessing",\n)\n',
    encoding="utf-8"
)
print("✓ wrote config.py")


✓ wrote config.py


In [5]:
# Optional: override the Gemini model from a Colab form field.
# Free-tier model availability varies by account — change this if the
# default in config.py isn't available to you.

GEMINI_MODEL = "gemini-3.6-flash"  # @param {type:"string"}
os.environ["GEMINI_MODEL"] = GEMINI_MODEL
print(f"Using model: {GEMINI_MODEL}")

Using model: gemini-3.6-flash


In [6]:
# Cell 6 — Gemini LLM wrapper
from pathlib import Path

Path("llm.py").parent.mkdir(parents=True, exist_ok=True)
Path("llm.py").write_text(
'"""\nllm.py\n======\nA thin, reusable wrapper around Google\'s `google-genai` SDK.\n\nThis module deliberately knows NOTHING about "agents", "tools", or\n"coding tasks" — it is a small, dependency-light interface for calling\nGemini and getting text back. Keeping it separate makes it trivial to\nswap the model, tune generation config, or mock it out for local unit\ntests that shouldn\'t burn free-tier quota.\n\nFREE-TIER QUOTA PROTECTION\n---------------------------\nThree things are layered on top of the raw API call, in this order:\n\n    1. Disk cache   -> identical prompts never hit the network twice.\n    2. Pacing       -> a minimum gap is enforced between live calls so\n                        we don\'t blow through a requests/minute limit.\n    3. Session cap  -> once GEMINI_SESSION_CALL_BUDGET live calls have\n                        been made (across the whole notebook run), any\n                        further call raises GeminiQuotaError so the\n                        agent can stop *gracefully* instead of crashing\n                        on a raw 429 partway through a demo.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Optional\n\nfrom config import (\n    GEMINI_MODEL,\n    GEMINI_MAX_OUTPUT_TOKENS,\n    MIN_SECONDS_BETWEEN_CALLS,\n    GEMINI_SESSION_CALL_BUDGET,\n    GEMINI_CACHE_ENABLED,\n    GEMINI_CACHE_DIR,\n)\n\n\nclass GeminiError(RuntimeError):\n    """Raised when a Gemini call fails after retries."""\n\n\nclass GeminiQuotaError(GeminiError):\n    """Raised when the local session call budget has been reached.\n\n    This is a *local* safety cap (see GEMINI_SESSION_CALL_BUDGET in\n    config.py), not necessarily the provider\'s own error — the goal is\n    to stop before we hit a real 429, so callers can catch this\n    specifically and end the run cleanly.\n    """\n\n\n@dataclass\nclass GeminiConfig:\n    model: str = GEMINI_MODEL\n    temperature: float = 0.2\n    max_output_tokens: int = GEMINI_MAX_OUTPUT_TOKENS\n\n\nclass GeminiLLM:\n    """\n    Reusable Gemini client.\n\n    Usage:\n        llm = GeminiLLM(api_key=GEMINI_API_KEY)\n        text = llm.generate("Write a haiku about recursion.")\n        data = llm.generate_json("Return {\'a\': 1} as JSON.")\n\n    The API key is passed in explicitly (never hard-coded) — in the\n    Colab notebook it is loaded from `google.colab.userdata`.\n\n    Pacing and the session call budget are tracked at the *class*\n    level, so they apply across every GeminiLLM instance created in a\n    notebook run (e.g. one per CodingAgent, one per demo task) — not\n    just within a single instance.\n    """\n\n    _last_call_ts: float = 0.0\n    _session_calls_used: int = 0\n\n    def __init__(\n        self,\n        api_key: str,\n        model: str = GEMINI_MODEL,\n        max_retries: int = 3,\n        retry_delay_seconds: float = 3.0,\n        min_seconds_between_calls: float = MIN_SECONDS_BETWEEN_CALLS,\n        session_call_budget: int = GEMINI_SESSION_CALL_BUDGET,\n        cache_enabled: bool = GEMINI_CACHE_ENABLED,\n        cache_dir: str = GEMINI_CACHE_DIR,\n    ) -> None:\n        if not api_key:\n            raise GeminiError(\n                "No Gemini API key was provided. Add GEMINI_API_KEY to "\n                "Colab Secrets (key icon in the left sidebar) and grant "\n                "this notebook access to it."\n            )\n        try:\n            from google import genai  # imported lazily so llm.py can be\n        except ImportError as exc:  # unit-tested without the package too\n            raise GeminiError(\n                "The \'google-genai\' package is not installed. "\n                "Run: pip install -q google-genai"\n            ) from exc\n\n        self._genai = genai\n        self._client = genai.Client(api_key=api_key)\n        self.model = model\n        self.max_retries = max_retries\n        self.retry_delay_seconds = retry_delay_seconds\n        self.min_seconds_between_calls = min_seconds_between_calls\n        self.session_call_budget = session_call_budget\n\n        self.cache_enabled = cache_enabled\n        self.cache_dir = Path(cache_dir)\n        if self.cache_enabled:\n            self.cache_dir.mkdir(parents=True, exist_ok=True)\n\n    # ------------------------------------------------------------------\n    # Session accounting (class-level, shared across instances)\n    # ------------------------------------------------------------------\n    @classmethod\n    def session_calls_used(cls) -> int:\n        """Live (non-cached) Gemini calls made so far in this notebook run."""\n        return cls._session_calls_used\n\n    @classmethod\n    def reset_session_budget(cls) -> None:\n        """Reset the shared call counter — mainly useful for local testing."""\n        cls._session_calls_used = 0\n        cls._last_call_ts = 0.0\n\n    # ------------------------------------------------------------------\n    # Disk cache — avoids re-spending quota on a prompt already run\n    # ------------------------------------------------------------------\n    def _cache_key(\n        self, prompt: str, temperature: float, max_output_tokens: int,\n        system_instruction: Optional[str],\n    ) -> str:\n        payload = json.dumps(\n            {\n                "model": self.model,\n                "prompt": prompt,\n                "temperature": temperature,\n                "max_output_tokens": max_output_tokens,\n                "system_instruction": system_instruction or "",\n            },\n            sort_keys=True,\n        )\n        return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n    def _read_cache(self, key: str) -> Optional[str]:\n        if not self.cache_enabled:\n            return None\n        path = self.cache_dir / f"{key}.txt"\n        if path.exists():\n            return path.read_text(encoding="utf-8")\n        return None\n\n    def _write_cache(self, key: str, text: str) -> None:\n        if not self.cache_enabled:\n            return\n        try:\n            (self.cache_dir / f"{key}.txt").write_text(text, encoding="utf-8")\n        except OSError:\n            pass  # cache is best-effort only — never fail the call over it\n\n    # ------------------------------------------------------------------\n    # Pacing + budget guards for live calls\n    # ------------------------------------------------------------------\n    def _respect_rate_limit(self) -> None:\n        elapsed = time.time() - GeminiLLM._last_call_ts\n        wait = self.min_seconds_between_calls - elapsed\n        if wait > 0:\n            time.sleep(wait)\n        GeminiLLM._last_call_ts = time.time()\n\n    def _check_budget(self) -> None:\n        if GeminiLLM._session_calls_used >= self.session_call_budget:\n            raise GeminiQuotaError(\n                f"Reached the local session budget of {self.session_call_budget} "\n                "live Gemini calls for this notebook run. This is a safety cap "\n                "(GEMINI_SESSION_CALL_BUDGET in config.py) meant to stop the "\n                "agent gracefully before your free-tier key returns a raw 429. "\n                "Increase the budget via the GEMINI_SESSION_CALL_BUDGET env "\n                "var, wait for your quota window to reset, or re-run — cached "\n                "prompts from earlier in this run cost nothing."\n            )\n\n    @staticmethod\n    def _is_rate_limit_error(exc: Exception) -> bool:\n        msg = str(exc).lower()\n        return any(\n            token in msg\n            for token in ("429", "resource_exhausted", "rate limit", "quota")\n        )\n\n    def _backoff_delay(self, exc: Exception, attempt: int) -> float:\n        base = self.retry_delay_seconds * (2 ** attempt)\n        if self._is_rate_limit_error(exc):\n            # Rate-limit errors need noticeably longer waits than a plain\n            # transient network hiccup.\n            base = max(base, 20.0 * (attempt + 1))\n        return base\n\n    # ------------------------------------------------------------------\n    # Core text generation\n    # ------------------------------------------------------------------\n    def generate(\n        self,\n        prompt: str,\n        temperature: float = 0.2,\n        max_output_tokens: int = GEMINI_MAX_OUTPUT_TOKENS,\n        system_instruction: Optional[str] = None,\n    ) -> str:\n        """\n        Send a single prompt to Gemini and return the text response.\n\n        Cache hits skip the network entirely (no pacing delay, no budget\n        cost). Live calls are paced and budget-checked, then retried a\n        small number of times on transient errors — with longer backoff\n        specifically for rate-limit errors — before raising GeminiError.\n        """\n        cache_key = self._cache_key(prompt, temperature, max_output_tokens, system_instruction)\n        cached = self._read_cache(cache_key)\n        if cached is not None:\n            return cached\n\n        self._check_budget()\n\n        last_error: Optional[Exception] = None\n\n        for attempt in range(self.max_retries + 1):\n            try:\n                self._respect_rate_limit()\n\n                config = {\n                    "temperature": temperature,\n                    "max_output_tokens": max_output_tokens,\n                }\n                if system_instruction:\n                    config["system_instruction"] = system_instruction\n\n                response = self._client.models.generate_content(\n                    model=self.model,\n                    contents=prompt,\n                    config=config,\n                )\n                GeminiLLM._session_calls_used += 1\n\n                text = getattr(response, "text", None)\n                if not text:\n                    raise GeminiError("Gemini returned an empty response.")\n\n                self._write_cache(cache_key, text)\n                return text\n            except Exception as exc:  # noqa: BLE001 - surface as GeminiError\n                last_error = exc\n                is_last_attempt = attempt == self.max_retries\n                if is_last_attempt:\n                    break\n                time.sleep(self._backoff_delay(exc, attempt))\n\n        raise GeminiError(\n            f"Gemini call failed after {self.max_retries + 1} attempt(s) "\n            f"using model \'{self.model}\': {last_error}"\n        )\n\n    # ------------------------------------------------------------------\n    # JSON-structured generation\n    # ------------------------------------------------------------------\n    def generate_json(\n        self,\n        prompt: str,\n        temperature: float = 0.1,\n        max_output_tokens: int = GEMINI_MAX_OUTPUT_TOKENS,\n    ) -> dict:\n        """\n        Ask Gemini for a JSON object and parse it robustly.\n\n        LLMs frequently wrap JSON in Markdown fences (```json ... ```)\n        or add stray prose — this method strips that before parsing,\n        and raises a clear GeminiError if parsing still fails.\n        """\n        strict_prompt = (\n            f"{prompt}\\n\\n"\n            "Respond with ONLY a valid JSON object. "\n            "Do not include Markdown code fences, explanations, or any "\n            "text before or after the JSON."\n        )\n        raw = self.generate(\n            strict_prompt,\n            temperature=temperature,\n            max_output_tokens=max_output_tokens,\n        )\n        cleaned = strip_markdown_fence(raw)\n        try:\n            return json.loads(cleaned)\n        except json.JSONDecodeError as exc:\n            raise GeminiError(\n                f"Gemini did not return valid JSON.\\nRaw response:\\n{raw}"\n            ) from exc\n\n\ndef strip_markdown_fence(text: str) -> str:\n    """\n    Remove surrounding Markdown code fences (```python, ```json, ``` etc.)\n    from an LLM response and return the inner content, trimmed.\n\n    Handles:\n      - ```python\\\\n<code>\\\\n```\n      - ```json\\\\n{...}\\\\n```\n      - ```\\\\n<code>\\\\n```\n      - plain text with no fences at all (returned unchanged, trimmed)\n    """\n    if text is None:\n        return ""\n\n    stripped = text.strip()\n\n    if not stripped.startswith("```"):\n        return stripped\n\n    lines = stripped.splitlines()\n\n    # Drop the opening fence line (``` or ```python / ```json / etc.)\n    if lines and lines[0].startswith("```"):\n        lines = lines[1:]\n\n    # Drop a trailing fence line, if present.\n    if lines and lines[-1].strip().startswith("```"):\n        lines = lines[:-1]\n\n    return "\\n".join(lines).strip()\n',
    encoding="utf-8"
)
print("✓ wrote llm.py")


✓ wrote llm.py


In [7]:
# Cells 7-13 — Agent state lives in coding_agent.py; tools (code gen, test gen, safe executor, test executor, failure analyzer/fixer) live here in tools.py
from pathlib import Path

Path("tools.py").parent.mkdir(parents=True, exist_ok=True)
Path("tools.py").write_text(
'"""\ntools.py\n========\nThe Coding Agent\'s tool system.\n\nEach tool is a plain Python callable/class with a single, well-defined\njob. Tools are deliberately "dumb" — they don\'t decide *when* to run,\nthey just do one thing well when the agent (coding_agent.py) invokes\nthem. This separation is what makes the "reasoning vs. execution"\ndistinction visible in the architecture:\n\n    reasoning/decision  -> coding_agent.py (the loop)\n    tool execution      -> tools.py (this file)\n\nSECURITY NOTE\n-------------\n`CodeExecutor` and `TestExecutor` run LLM-generated Python source with\n`subprocess.run(...)` inside a temporary directory, with a timeout and\ncaptured output. This is a *controlled educational demonstration only*\n— it is NOT a hardened security sandbox. See the README\'s "Security\nConsiderations" section before using this with untrusted input.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport subprocess\nimport sys\nimport tempfile\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Optional\n\nfrom llm import GeminiLLM, strip_markdown_fence\nfrom config import EXECUTION_TIMEOUT, GEMINI_TEMPERATURE_CODEGEN, GEMINI_TEMPERATURE_FIX\n\n\n# ===========================================================================\n# Tool 1 — Code Generator\n# ===========================================================================\nclass CodeGeneratorTool:\n    """Generates a Python implementation from task + requirements + plan."""\n\n    def __init__(self, llm: GeminiLLM) -> None:\n        self.llm = llm\n\n    def run(self, task: str, requirements: list[str], plan: list[str]) -> str:\n        prompt = f"""You are an expert Python engineer. Write a complete,\nrunnable Python module that implements the following task.\n\nTASK:\n{task}\n\nREQUIREMENTS:\n{_bullet(requirements)}\n\nIMPLEMENTATION PLAN:\n{_bullet(plan)}\n\nRules:\n- Return ONLY Python source code, no explanations.\n- Wrap the code in a single ```python fenced block.\n- Use functions and/or classes as appropriate, with type hints and\n  docstrings.\n- Include basic error handling where it matters.\n- The module must be self-contained (standard library only, unless the\n  task clearly requires a specific well-known package).\n- If the task implies a CLI, guard the entry point with\n  `if __name__ == "__main__":`.\n"""\n        raw = self.llm.generate(prompt, temperature=GEMINI_TEMPERATURE_CODEGEN)\n        code = strip_markdown_fence(raw)\n        if not code.strip():\n            raise ValueError("Code generation returned empty output.")\n        return code\n\n\n# ===========================================================================\n# Tool 2 — Test Generator\n# ===========================================================================\nclass TestGeneratorTool:\n    """Generates `unittest`-based tests for a given code module."""\n\n    def __init__(self, llm: GeminiLLM) -> None:\n        self.llm = llm\n\n    def run(self, task: str, code: str) -> str:\n        prompt = f"""You are an expert Python test engineer. Write\n`unittest`-based tests for the module below.\n\nTASK CONTEXT:\n{task}\n\nMODULE UNDER TEST (this exact source will be saved as `solution.py`,\nso import from it with `from solution import ...`):\n```python\n{code}\n```\n\nRules:\n- Return ONLY Python source code, no explanations.\n- Wrap the code in a single ```python fenced block.\n- Use the standard library `unittest` module only.\n- Import the module under test with `from solution import ...`.\n- Cover normal cases, edge cases, and at least one failure/invalid case.\n- End the file with:\n  if __name__ == "__main__":\n      unittest.main()\n"""\n        raw = self.llm.generate(prompt, temperature=GEMINI_TEMPERATURE_CODEGEN)\n        tests = strip_markdown_fence(raw)\n        if not tests.strip():\n            raise ValueError("Test generation returned empty output.")\n        return tests\n\n\n# ===========================================================================\n# Shared execution result type\n# ===========================================================================\n@dataclass\nclass ExecutionResult:\n    success: bool\n    returncode: Optional[int]\n    stdout: str\n    stderr: str\n    timed_out: bool = False\n    error: str = ""\n\n    def summary(self) -> str:\n        if self.timed_out:\n            return f"TIMEOUT after {EXECUTION_TIMEOUT}s"\n        if self.error:\n            return f"ERROR: {self.error}"\n        return f"exit code {self.returncode}"\n\n\n# ===========================================================================\n# Tool 3 — Code Executor\n# ===========================================================================\nclass CodeExecutor:\n    """\n    Runs a Python source string in an isolated temporary directory.\n\n    This is a *controlled educational demonstration*, not a security\n    sandbox. It only ever invokes the current Python interpreter on a\n    file we wrote ourselves (`python solution.py`) — it never executes\n    arbitrary shell commands produced by the LLM.\n    """\n\n    def __init__(self, timeout: int = EXECUTION_TIMEOUT) -> None:\n        self.timeout = timeout\n\n    def run(self, code: str, filename: str = "solution.py") -> ExecutionResult:\n        with tempfile.TemporaryDirectory(prefix="coding_agent_") as tmpdir:\n            file_path = Path(tmpdir) / filename\n            file_path.write_text(code, encoding="utf-8")\n            return self._execute([sys.executable, str(file_path)], cwd=tmpdir)\n\n    def _execute(self, cmd: list[str], cwd: str) -> ExecutionResult:\n        try:\n            proc = subprocess.run(\n                cmd,\n                cwd=cwd,\n                capture_output=True,\n                text=True,\n                timeout=self.timeout,\n            )\n            return ExecutionResult(\n                success=proc.returncode == 0,\n                returncode=proc.returncode,\n                stdout=proc.stdout,\n                stderr=proc.stderr,\n            )\n        except subprocess.TimeoutExpired as exc:\n            return ExecutionResult(\n                success=False,\n                returncode=None,\n                stdout=exc.stdout or "",\n                stderr=exc.stderr or "",\n                timed_out=True,\n            )\n        except Exception as exc:  # noqa: BLE001\n            return ExecutionResult(\n                success=False,\n                returncode=None,\n                stdout="",\n                stderr="",\n                error=str(exc),\n            )\n        # Temporary directory and all files inside it are removed\n        # automatically when the `with` block exits.\n\n\n# ===========================================================================\n# Tool 4 — Test Executor\n# ===========================================================================\nclass TestExecutor:\n    """\n    Runs `unittest` tests against a code module in an isolated temp dir.\n\n    Writes `solution.py` and `test_solution.py` side by side, then runs\n    `python -m unittest test_solution.py -v` and parses a pass/fail\n    summary out of the captured output.\n    """\n\n    def __init__(self, timeout: int = EXECUTION_TIMEOUT) -> None:\n        self.timeout = timeout\n        self._executor = CodeExecutor(timeout=timeout)\n\n    def run(self, code: str, tests: str) -> "TestRunResult":\n        with tempfile.TemporaryDirectory(prefix="coding_agent_tests_") as tmpdir:\n            (Path(tmpdir) / "solution.py").write_text(code, encoding="utf-8")\n            (Path(tmpdir) / "test_solution.py").write_text(tests, encoding="utf-8")\n\n            exec_result = self._executor._execute(\n                [sys.executable, "-m", "unittest", "test_solution.py", "-v"],\n                cwd=tmpdir,\n            )\n            passed, failed, errors = _parse_unittest_output(exec_result.stdout, exec_result.stderr)\n            return TestRunResult(\n                execution=exec_result,\n                tests_passed=passed,\n                tests_failed=failed,\n                tests_errored=errors,\n            )\n\n\n@dataclass\nclass TestRunResult:\n    execution: ExecutionResult\n    tests_passed: int = 0\n    tests_failed: int = 0\n    tests_errored: int = 0\n\n    @property\n    def all_passed(self) -> bool:\n        return (\n            self.execution.success\n            and self.tests_failed == 0\n            and self.tests_errored == 0\n        )\n\n    def summary(self) -> str:\n        return (\n            f"passed={self.tests_passed} failed={self.tests_failed} "\n            f"errors={self.tests_errored} ({self.execution.summary()})"\n        )\n\n\ndef _parse_unittest_output(stdout_text: str, stderr_text: str) -> tuple[int, int, int]:\n    """\n    `unittest -v` normally writes its progress/summary lines to stderr,\n    but a module-level failure during test *collection* (e.g. a\n    SyntaxError, ImportError, or ModuleNotFoundError in solution.py or\n    test_solution.py) can surface as a plain traceback that is not\n    formatted as a per-test "... ok/FAIL/ERROR" line, and depending on\n    how it is raised may show up on stdout rather than stderr. Scan\n    both streams so collection-time failures are always counted as an\n    error instead of silently reporting zero failures. This is\n    intentionally simple string parsing, not a full test-report parser.\n    """\n    passed = failed = errored = 0\n    for raw_line in stderr_text.splitlines() + stdout_text.splitlines():\n        line = raw_line.strip()\n        if line.endswith("... ok"):\n            passed += 1\n        elif line.endswith("... FAIL"):\n            failed += 1\n        elif line.endswith("... ERROR"):\n            errored += 1\n        elif any(\n            marker in line\n            for marker in ("SyntaxError", "ImportError", "ModuleNotFoundError")\n        ):\n            errored += 1\n    return passed, failed, errored\n\n\n# ===========================================================================\n# Tool 5 — Code Analyzer / Fixer\n# ===========================================================================\nclass CodeFixerTool:\n    """\n    Sends failure details to Gemini and asks for a corrected full module.\n    Used for both "code raised an error" and "tests failed" cases.\n    """\n\n    def __init__(self, llm: GeminiLLM) -> None:\n        self.llm = llm\n\n    def diagnose_and_fix(\n        self,\n        task: str,\n        code: str,\n        tests: str,\n        error_output: str,\n    ) -> tuple[dict, str]:\n        """\n        Returns (diagnosis_dict, fixed_code_str).\n\n        A single Gemini call is used to both diagnose the failure and\n        produce the fix, to conserve free-tier quota. The diagnosis is\n        requested as a JSON preamble embedded in a larger response, but\n        to keep parsing simple and robust we issue two lightweight\n        prompts sharing the same context instead of one fragile combined\n        format — this still counts as ONE logical "fix step" in the\n        agent\'s call budget documentation, but uses `generate_json` +\n        `generate` under the hood. See README\'s Free-Tier Optimization\n        section for how call budget is tracked.\n        """\n        diagnosis_prompt = f"""A Python solution failed. Diagnose the\nroot cause briefly.\n\nTASK:\n{task}\n\nCODE:\n```python\n{code}\n```\n\nTESTS:\n```python\n{tests}\n```\n\nFAILURE / ERROR OUTPUT:\n```\n{error_output}\n```\n\nReturn a JSON object with exactly these keys:\n{{"problem": "...", "cause": "...", "fix_strategy": "..."}}\n"""\n        diagnosis = self.llm.generate_json(diagnosis_prompt)\n\n        fix_prompt = f"""Fix the Python module below so it satisfies the\ntask and passes the tests. Apply this fix strategy: {diagnosis.get(\'fix_strategy\', \'N/A\')}\n\nTASK:\n{task}\n\nCURRENT CODE:\n```python\n{code}\n```\n\nTESTS IT MUST PASS:\n```python\n{tests}\n```\n\nFAILURE / ERROR OUTPUT:\n```\n{error_output}\n```\n\nReturn ONLY the complete corrected Python module in a single ```python\nfenced block. Do not include explanations.\n"""\n        raw = self.llm.generate(fix_prompt, temperature=GEMINI_TEMPERATURE_FIX)\n        fixed_code = strip_markdown_fence(raw)\n        if not fixed_code.strip():\n            raise ValueError("Code fixer returned empty output.")\n        return diagnosis, fixed_code\n\n\n# ===========================================================================\n# Small shared helpers\n# ===========================================================================\ndef _bullet(items: list[str]) -> str:\n    if not items:\n        return "- (none specified)"\n    return "\\n".join(f"- {item}" for item in items)\n\n\n# ===========================================================================\n# Tool registry (used by the agent + exercised by unit tests)\n# ===========================================================================\ndef build_tool_registry(llm: GeminiLLM) -> dict:\n    """\n    Constructs and returns all tools keyed by name. Keeping this in one\n    place makes it easy to see (and test) exactly what tools the agent\n    has available.\n    """\n    return {\n        "code_generator": CodeGeneratorTool(llm),\n        "test_generator": TestGeneratorTool(llm),\n        "code_executor": CodeExecutor(),\n        "test_executor": TestExecutor(),\n        "code_fixer": CodeFixerTool(llm),\n    }\n',
    encoding="utf-8"
)
print("✓ wrote tools.py")

✓ wrote tools.py


In [8]:
# Cell 14 — Agent state (CodingState), actions (Action enum), and the bounded agent loop (CodingAgent)
from pathlib import Path

Path("coding_agent.py").parent.mkdir(parents=True, exist_ok=True)
Path("coding_agent.py").write_text(
'"""\ncoding_agent.py\n================\nThe agent itself: explicit state, a small set of named actions, and a\nbounded loop that decides what to do next based on the current state.\n\n    reasoning/decision   -> CodingAgent._decide_next_action()\n    tool execution       -> CodingAgent._execute_action() (delegates to tools.py)\n\nThis file has NO knowledge of Colab, files, or ZIPs — it\'s pure agent\nlogic so it can be unit-tested and reused anywhere.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nfrom pathlib import Path\nfrom typing import Optional\n\nfrom config import MAX_ITERATIONS, GEMINI_TEMPERATURE_PLANNING\nfrom llm import GeminiLLM, GeminiError, GeminiQuotaError\nfrom tools import build_tool_registry, TestRunResult\n\n\n# ===========================================================================\n# Actions\n# ===========================================================================\nclass Action(str, Enum):\n    PLAN = "PLAN"\n    GENERATE_CODE = "GENERATE_CODE"\n    GENERATE_TESTS = "GENERATE_TESTS"\n    RUN_TESTS = "RUN_TESTS"\n    ANALYZE_FAILURE = "ANALYZE_FAILURE"\n    FIX_CODE = "FIX_CODE"\n    FINISH = "FINISH"\n\n\n# ===========================================================================\n# Agent state\n# ===========================================================================\n@dataclass\nclass CodingState:\n    """Everything the agent knows about the current task, at any point\n    in the loop. This object is mutated in place as the loop progresses\n    and is what makes the agent\'s behavior "agentic" rather than a\n    single stateless prompt -> response call."""\n\n    task: str\n    requirements: list = field(default_factory=list)\n    plan: list = field(default_factory=list)\n    code: str = ""\n    tests: str = ""\n    execution_results: list = field(default_factory=list)  # list[dict]\n    errors: list = field(default_factory=list)              # list[str]\n    fixes: list = field(default_factory=list)                # list[dict]\n    iteration: int = 0\n    success: bool = False\n    final_output: str = ""\n    last_test_result: Optional[TestRunResult] = field(default=None, repr=False)\n    gemini_calls_used: int = 0\n    stopped_reason: str = ""  # set if the loop had to stop early (quota/API)\n\n    def log_execution(self, label: str, summary: str) -> None:\n        self.execution_results.append({"step": label, "summary": summary})\n\n\n# ===========================================================================\n# The Agent\n# ===========================================================================\nclass CodingAgent:\n    """\n    Orchestrates the agentic coding loop:\n\n        while not success and iteration < MAX_ITERATIONS:\n            analyze_state()\n            action = decide_next_action()\n            execute_action(action)\n            record_observation()\n            update_state()\n\n    If a Gemini call fails permanently (including hitting the local\n    free-tier session budget), the loop stops *gracefully* — whatever\n    partial progress exists is still returned and reported cleanly,\n    instead of raising and leaving a raw traceback in the notebook.\n    """\n\n    def __init__(\n        self,\n        api_key: str,\n        model: Optional[str] = None,\n        max_iterations: int = MAX_ITERATIONS,\n        verbose: bool = True,\n    ) -> None:\n        self.llm = GeminiLLM(api_key=api_key, model=model) if model else GeminiLLM(api_key=api_key)\n        self.tools = build_tool_registry(self.llm)\n        self.max_iterations = max_iterations\n        self.verbose = verbose\n\n    # ------------------------------------------------------------------\n    # Public entry point\n    # ------------------------------------------------------------------\n    def run(self, task: str) -> CodingState:\n        state = CodingState(task=task)\n        self._log("=" * 56)\n        self._log("🤖 AGENTIC CODING AGENT")\n        self._log("=" * 56)\n        self._log(f"\\n🎯 Task:\\n{task}\\n")\n\n        try:\n            while not state.success and state.iteration < self.max_iterations:\n                state.iteration += 1\n                self._log(f"\\n--- Iteration {state.iteration}/{self.max_iterations} ---")\n\n                action = self._decide_next_action(state)\n                self._execute_action(action, state)\n\n                # Register success the moment tests actually pass, even if\n                # this was the last allowed iteration — otherwise a task\n                # that truly succeeded on its final try would incorrectly\n                # be reported as "NEEDS REVIEW" simply because the loop\n                # never got a further turn to reach Action.FINISH.\n                if state.last_test_result is not None and state.last_test_result.all_passed:\n                    state.success = True\n\n                if action == Action.FINISH:\n                    break\n        except GeminiQuotaError as exc:\n            state.stopped_reason = f"Session call budget reached: {exc}"\n            self._log(f"\\n⏸ Pausing — {state.stopped_reason}")\n        except GeminiError as exc:\n            state.stopped_reason = f"Gemini call failed: {exc}"\n            self._log(f"\\n⏸ Pausing — {state.stopped_reason}")\n\n        self._finalize(state)\n        return state\n\n    # ------------------------------------------------------------------\n    # Reasoning / decision layer\n    # ------------------------------------------------------------------\n    def _decide_next_action(self, state: CodingState) -> Action:\n        """Pure decision logic based on current state — no tool calls\n        happen here. This is what keeps \'deciding what to do\' separate\n        from \'doing it\'."""\n        if not state.plan:\n            return Action.PLAN\n        if not state.code:\n            return Action.GENERATE_CODE\n        if not state.tests:\n            return Action.GENERATE_TESTS\n        if state.last_test_result is None:\n            return Action.RUN_TESTS\n        if state.last_test_result.all_passed:\n            return Action.FINISH\n        if state.iteration >= self.max_iterations:\n            return Action.FINISH\n        # Tests exist and failed -> diagnose + fix, then re-test\n        if not state.fixes or len(state.fixes) < state.iteration - 1:\n            return Action.ANALYZE_FAILURE\n        return Action.RUN_TESTS\n\n    # ------------------------------------------------------------------\n    # Execution layer — delegates to tools.py, never reasons\n    # ------------------------------------------------------------------\n    def _execute_action(self, action: Action, state: CodingState) -> None:\n        handler = {\n            Action.PLAN: self._act_plan,\n            Action.GENERATE_CODE: self._act_generate_code,\n            Action.GENERATE_TESTS: self._act_generate_tests,\n            Action.RUN_TESTS: self._act_run_tests,\n            Action.ANALYZE_FAILURE: self._act_analyze_and_fix,\n            Action.FINISH: self._act_finish,\n        }[action]\n        handler(state)\n\n    # -- Step 1: Understand requirements + plan (1 Gemini call) --------\n    def _act_plan(self, state: CodingState) -> None:\n        self._log("\\n🧠 Step — Understanding requirements & creating plan")\n        prompt = f"""Analyze this coding task and produce a JSON object\nwith the requirements and an implementation plan.\n\nTASK:\n{state.task}\n\nReturn JSON with exactly these keys:\n{{\n  "requirements": ["...", "..."],\n  "plan": ["...", "..."]\n}}\n"""\n        result = self.llm.generate_json(prompt, temperature=GEMINI_TEMPERATURE_PLANNING)\n        state.gemini_calls_used += 1\n        state.requirements = result.get("requirements", [])\n        state.plan = result.get("plan", [])\n        self._log(f"✓ {len(state.requirements)} requirements identified")\n        self._log(f"✓ Plan created ({len(state.plan)} steps)")\n        state.log_execution("plan", f"{len(state.requirements)} requirements, {len(state.plan)} plan steps")\n\n    # -- Step 2: Generate code (1 Gemini call) --------------------------\n    def _act_generate_code(self, state: CodingState) -> None:\n        self._log("\\n💻 Step — Generating code")\n        state.code = self.tools["code_generator"].run(\n            task=state.task, requirements=state.requirements, plan=state.plan\n        )\n        state.gemini_calls_used += 1\n        self._log("✓ Code generated")\n        state.log_execution("generate_code", f"{len(state.code.splitlines())} lines generated")\n\n    # -- Step 3: Generate tests (part of the code-gen call budget) -----\n    def _act_generate_tests(self, state: CodingState) -> None:\n        self._log("\\n🧪 Step — Generating tests")\n        state.tests = self.tools["test_generator"].run(task=state.task, code=state.code)\n        state.gemini_calls_used += 1\n        self._log("✓ Tests generated")\n        state.log_execution("generate_tests", f"{len(state.tests.splitlines())} lines generated")\n\n    # -- Step 4/8: Run tests (local, no Gemini call) --------------------\n    def _act_run_tests(self, state: CodingState) -> None:\n        self._log("\\n▶ Step — Running tests")\n        result = self.tools["test_executor"].run(state.code, state.tests)\n        state.last_test_result = result\n        if result.all_passed:\n            self._log(f"✓ All tests passed ({result.summary()})")\n        else:\n            self._log(f"✗ Tests failed ({result.summary()})")\n            combined_output = (result.execution.stdout + "\\n" + result.execution.stderr).strip()\n            state.errors.append(combined_output or result.execution.summary())\n        state.log_execution("run_tests", result.summary())\n\n    # -- Step 6/7: Diagnose + fix (1 Gemini call) ------------------------\n    def _act_analyze_and_fix(self, state: CodingState) -> None:\n        self._log("\\n🔍 Step — Diagnosing failure")\n        error_output = state.errors[-1] if state.errors else "(no captured output)"\n        diagnosis, fixed_code = self.tools["code_fixer"].diagnose_and_fix(\n            task=state.task, code=state.code, tests=state.tests, error_output=error_output\n        )\n        state.gemini_calls_used += 1\n        self._log(f"✓ Root cause identified: {diagnosis.get(\'problem\', \'unknown\')}")\n        state.fixes.append(diagnosis)\n\n        self._log("\\n🔧 Step — Fixing code")\n        state.code = fixed_code\n        state.last_test_result = None  # force a re-test on the next iteration\n        self._log("✓ Updated implementation")\n        state.log_execution("fix_code", diagnosis.get("fix_strategy", "applied fix"))\n\n    # -- Finalization ------------------------------------------------------\n    def _act_finish(self, state: CodingState) -> None:\n        if state.last_test_result is not None and state.last_test_result.all_passed:\n            state.success = True\n\n    def _finalize(self, state: CodingState) -> None:\n        self._log("\\n" + "=" * 56)\n        if state.stopped_reason:\n            status = "PAUSED — QUOTA/RATE LIMIT"\n            self._log("⏸ STOPPED EARLY TO PROTECT YOUR FREE-TIER QUOTA")\n            self._log(f"   {state.stopped_reason}")\n            self._log("   Whatever code/tests were produced before this point are")\n            self._log("   shown below. Re-run later (or raise the session budget /")\n            self._log("   MIN_SECONDS_BETWEEN_CALLS in config.py) once quota resets.")\n        elif state.success:\n            status = "SUCCESS"\n            self._log("✅ CODING TASK COMPLETED")\n        else:\n            status = "NEEDS REVIEW"\n            self._log("⚠ Maximum iterations reached.")\n            self._log("The generated solution still requires review.")\n        self._log("=" * 56)\n\n        state.final_output = state.code\n        budget = self.llm.session_call_budget\n        used = GeminiLLM.session_calls_used()\n        self._log(f"\\n📊 Live Gemini calls this session: {used}/{budget}")\n        state.log_execution("finish", f"status={status}, gemini_calls={state.gemini_calls_used}")\n\n    # ------------------------------------------------------------------\n    def _log(self, message: str) -> None:\n        if self.verbose:\n            print(message)\n\n\n# ===========================================================================\n# Convenience: save final code to disk\n# ===========================================================================\ndef save_final_code(state: CodingState, output_path: str = "final_solution.py") -> Path:\n    path = Path(output_path)\n    path.write_text(state.final_output or state.code, encoding="utf-8")\n    return path\n\n\ndef print_final_report(state: CodingState) -> None:\n    if state.stopped_reason:\n        status = "PAUSED — QUOTA/RATE LIMIT"\n    elif state.success:\n        status = "SUCCESS"\n    else:\n        status = "NEEDS REVIEW"\n    result = state.last_test_result\n    print("\\n" + "=" * 56)\n    print("FINAL RESULT")\n    print("=" * 56)\n    print(f"\\nTask:\\n{state.task}\\n")\n    print(f"Iterations: {state.iteration}")\n    print(f"Gemini calls used (this task): {state.gemini_calls_used}")\n    print(f"Gemini calls used (this session): {GeminiLLM.session_calls_used()}")\n    if result:\n        print(f"Tests passed: {result.tests_passed}")\n        print(f"Tests failed: {result.tests_failed + result.tests_errored}")\n    print(f"\\nStatus: {status}")\n    if state.stopped_reason:\n        print(f"Reason: {state.stopped_reason}")\n    if state.code:\n        print("\\n--- Final Code (partial or complete, see status above) ---\\n")\n        print(state.final_output or state.code)\n    else:\n        print("\\n(No code was generated before stopping.)")\n',
    encoding="utf-8"
)
print("✓ wrote coding_agent.py")


✓ wrote coding_agent.py


In [9]:
# Reload modules freshly now that all files are written to disk.
import importlib
for mod_name in ["config", "llm", "tools", "coding_agent"]:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])
    else:
        importlib.import_module(mod_name)

from config import MAX_ITERATIONS, EXECUTION_TIMEOUT, GEMINI_MODEL
from llm import GeminiLLM, GeminiError, strip_markdown_fence
from tools import (
    CodeGeneratorTool, TestGeneratorTool, CodeExecutor, TestExecutor,
    CodeFixerTool, build_tool_registry,
)
from coding_agent import (
    CodingState, Action, CodingAgent, save_final_code, print_final_report,
)

print("\u2713 modules loaded")
print(f"MAX_ITERATIONS={MAX_ITERATIONS}  EXECUTION_TIMEOUT={EXECUTION_TIMEOUT}s  GEMINI_MODEL={GEMINI_MODEL}")

✓ modules loaded
MAX_ITERATIONS=6  EXECUTION_TIMEOUT=10s  GEMINI_MODEL=gemini-3.6-flash


In [10]:
# Build the agent instance (requires GEMINI_API_KEY from Cell 4).
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in Colab Secrets first (see Cell 4)."

agent = CodingAgent(api_key=GEMINI_API_KEY, model=GEMINI_MODEL, max_iterations=MAX_ITERATIONS)
print("\u2713 CodingAgent ready")

✓ CodingAgent ready


In [13]:
# Enter your own task and run the agent on it.
custom_task = "Create a Python function that checks whether a number is even or odd."  # @param {type:"string"}

custom_state = agent.run(custom_task)

🤖 AGENTIC CODING AGENT

🎯 Task:
Create a Python function that checks whether a number is even or odd.


--- Iteration 1/6 ---

🧠 Step — Understanding requirements & creating plan
✓ 4 requirements identified
✓ Plan created (5 steps)

--- Iteration 2/6 ---

💻 Step — Generating code
✓ Code generated

--- Iteration 3/6 ---

🧪 Step — Generating tests
✓ Tests generated

--- Iteration 4/6 ---

▶ Step — Running tests
✓ All tests passed (passed=1 failed=0 errors=0 (exit code 0))

✅ CODING TASK COMPLETED

📊 Live Gemini calls this session: 5/20


In [14]:
# Display the final generated code for the custom task run above.
print("=" * 56)
print("FINAL GENERATED CODE")
print("=" * 56)
print(custom_state.final_output or custom_state.code)

FINAL GENERATED CODE
"""Module to determine whether a number is even or odd."""

import sys
from typing import Union


def is_even_or_odd(number: Union[int, float]) -> str:
    """Determine whether a given number is even or odd.

    Args:
        number (Union[int, float]): The number to check. Must be an integer or
            a float representing a whole number.

    Returns:
        str: 'Even' if the number is even, 'Odd' if the number is odd.

    Raises:
        TypeError: If the input is not a numeric type (int or float).
        ValueError: If a float input is not a whole integer.
    """
    if isinstance(number, bool) or not isinstance(number, (int, float)):
        raise TypeError(
            f"Expected int or float, got {type(number).__name__}"
        )

    if isinstance(number, float):
        if not number.is_integer():
            raise ValueError(
                f"Cannot determine parity of non-integer float: {number}"
            )
        number = int(number)

  

In [15]:
# Display the full execution trace + test results for the custom task.
print_final_report(custom_state)

print("\n--- Execution trace ---")
for entry in custom_state.execution_results:
    print(f"  [{entry['step']}] {entry['summary']}")

if custom_state.fixes:
    print("\n--- Diagnoses applied ---")
    for i, fix in enumerate(custom_state.fixes, 1):
        print(f"  {i}. {fix.get('problem')} -> {fix.get('fix_strategy')}")


FINAL RESULT

Task:
Create a Python function that checks whether a number is even or odd.

Iterations: 4
Gemini calls used (this task): 3
Gemini calls used (this session): 5
Tests passed: 1
Tests failed: 0

Status: SUCCESS

--- Final Code (partial or complete, see status above) ---

"""Module to determine whether a number is even or odd."""

import sys
from typing import Union


def is_even_or_odd(number: Union[int, float]) -> str:
    """Determine whether a given number is even or odd.

    Args:
        number (Union[int, float]): The number to check. Must be an integer or
            a float representing a whole number.

    Returns:
        str: 'Even' if the number is even, 'Odd' if the number is odd.

    Raises:
        TypeError: If the input is not a numeric type (int or float).
        ValueError: If a float input is not a whole integer.
    """
    if isinstance(number, bool) or not isinstance(number, (int, float)):
        raise TypeError(
            f"Expected int or flo

In [16]:
# Cell 20 — Local unit tests (no Gemini calls)
from pathlib import Path

Path("tests/test_tools.py").parent.mkdir(parents=True, exist_ok=True)
Path("tests/test_tools.py").write_text(
'"""\ntests/test_tools.py\n====================\nPure, local unit tests that do NOT call the Gemini API. These are safe\nto run repeatedly without touching the free-tier quota.\n\nCovers:\n    1. Markdown code-fence stripping\n    2. Temporary workspace creation / cleanup\n    3. Code execution (success + failure)\n    4. Timeout handling\n    5. Agent state initialization\n    6. Tool registry construction\n\nRun with:\n    python -m unittest tests/test_tools.py -v\n(from the project root, with coding_agent.py / tools.py / llm.py /\nconfig.py importable — e.g. `PYTHONPATH=. python -m unittest discover`)\n"""\n\nfrom __future__ import annotations\n\nimport sys\nimport unittest\nfrom pathlib import Path\n\n# Make the project root importable when running this file directly.\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\n\nfrom llm import strip_markdown_fence\nfrom tools import CodeExecutor, TestExecutor, build_tool_registry\nfrom coding_agent import CodingState, Action\n\n\nclass TestMarkdownFenceStripping(unittest.TestCase):\n    def test_strips_python_fence(self):\n        raw = "```python\\nprint(\'hi\')\\n```"\n        self.assertEqual(strip_markdown_fence(raw), "print(\'hi\')")\n\n    def test_strips_bare_fence(self):\n        raw = "```\\nx = 1\\n```"\n        self.assertEqual(strip_markdown_fence(raw), "x = 1")\n\n    def test_no_fence_returns_trimmed_text(self):\n        raw = "  x = 1  \\n"\n        self.assertEqual(strip_markdown_fence(raw), "x = 1")\n\n    def test_empty_input(self):\n        self.assertEqual(strip_markdown_fence(""), "")\n        self.assertEqual(strip_markdown_fence(None), "")\n\n\nclass TestCodeExecutor(unittest.TestCase):\n    def setUp(self):\n        self.executor = CodeExecutor(timeout=5)\n\n    def test_successful_execution(self):\n        code = "print(\'hello world\')"\n        result = self.executor.run(code)\n        self.assertTrue(result.success)\n        self.assertEqual(result.returncode, 0)\n        self.assertIn("hello world", result.stdout)\n\n    def test_runtime_error_is_captured_not_raised(self):\n        code = "raise ValueError(\'boom\')"\n        result = self.executor.run(code)\n        self.assertFalse(result.success)\n        self.assertNotEqual(result.returncode, 0)\n        self.assertIn("ValueError", result.stderr)\n\n    def test_timeout_handling(self):\n        executor = CodeExecutor(timeout=1)\n        code = "import time\\ntime.sleep(5)"\n        result = executor.run(code)\n        self.assertFalse(result.success)\n        self.assertTrue(result.timed_out)\n\n    def test_temporary_workspace_is_cleaned_up(self):\n        # The executor\'s tempdir is a context manager; after run()\n        # returns, nothing from it should remain on disk. We can\'t\n        # easily inspect the exact path, so instead we verify that two\n        # consecutive runs don\'t collide or leak state into each other.\n        code_a = "x = 1\\nprint(x)"\n        code_b = "print(\'no x defined here\')"\n        result_a = self.executor.run(code_a)\n        result_b = self.executor.run(code_b)\n        self.assertIn("1", result_a.stdout)\n        self.assertIn("no x defined here", result_b.stdout)\n\n\nclass TestTestExecutor(unittest.TestCase):\n    def test_passing_tests_are_detected(self):\n        code = (\n            "def add(a: int, b: int) -> int:\\n"\n            "    return a + b\\n"\n        )\n        tests = (\n            "import unittest\\n"\n            "from solution import add\\n\\n"\n            "class TestAdd(unittest.TestCase):\\n"\n            "    def test_add(self):\\n"\n            "        self.assertEqual(add(2, 3), 5)\\n\\n"\n            "if __name__ == \'__main__\':\\n"\n            "    unittest.main()\\n"\n        )\n        result = TestExecutor(timeout=5).run(code, tests)\n        self.assertTrue(result.all_passed)\n        self.assertEqual(result.tests_failed, 0)\n\n    def test_failing_tests_are_detected(self):\n        code = (\n            "def add(a: int, b: int) -> int:\\n"\n            "    return a - b  # intentionally wrong\\n"\n        )\n        tests = (\n            "import unittest\\n"\n            "from solution import add\\n\\n"\n            "class TestAdd(unittest.TestCase):\\n"\n            "    def test_add(self):\\n"\n            "        self.assertEqual(add(2, 3), 5)\\n\\n"\n            "if __name__ == \'__main__\':\\n"\n            "    unittest.main()\\n"\n        )\n        result = TestExecutor(timeout=5).run(code, tests)\n        self.assertFalse(result.all_passed)\n\n\nclass TestAgentState(unittest.TestCase):\n    def test_default_initialization(self):\n        state = CodingState(task="do something")\n        self.assertEqual(state.task, "do something")\n        self.assertEqual(state.requirements, [])\n        self.assertEqual(state.plan, [])\n        self.assertEqual(state.code, "")\n        self.assertEqual(state.tests, "")\n        self.assertEqual(state.iteration, 0)\n        self.assertFalse(state.success)\n        self.assertEqual(state.gemini_calls_used, 0)\n\n    def test_log_execution_appends_entry(self):\n        state = CodingState(task="t")\n        state.log_execution("plan", "3 requirements")\n        self.assertEqual(len(state.execution_results), 1)\n        self.assertEqual(state.execution_results[0]["step"], "plan")\n\n    def test_action_enum_has_expected_members(self):\n        expected = {\n            "PLAN", "GENERATE_CODE", "GENERATE_TESTS", "RUN_TESTS",\n            "ANALYZE_FAILURE", "FIX_CODE", "FINISH",\n        }\n        actual = {a.name for a in Action}\n        self.assertTrue(expected.issubset(actual))\n\n\nclass TestToolRegistry(unittest.TestCase):\n    def test_registry_contains_all_expected_tools(self):\n        # A dummy stand-in is fine here: tool constructors only store a\n        # reference to the LLM, they never call it at construction time,\n        # so no real API key or network access is needed for this test.\n        dummy_llm = object()\n        registry = build_tool_registry(dummy_llm)\n        expected_keys = {\n            "code_generator", "test_generator", "code_executor",\n            "test_executor", "code_fixer",\n        }\n        self.assertEqual(set(registry.keys()), expected_keys)\n\n\nif __name__ == "__main__":\n    unittest.main()\n',
    encoding="utf-8"
)
print("✓ wrote tests/test_tools.py")

✓ wrote tests/test_tools.py


In [19]:
# Run the local unit tests — these never call the Gemini API,
# so they're safe to re-run as often as you like.
!python /content/tests/test_tools.py

..............
----------------------------------------------------------------------
Ran 14 tests in 1.304s

OK


In [20]:
# Cell 21a — requirements.txt
from pathlib import Path

Path("requirements.txt").parent.mkdir(parents=True, exist_ok=True)
Path("requirements.txt").write_text(
'google-genai>=0.3.0\n',
    encoding="utf-8"
)
print("✓ wrote requirements.txt")

✓ wrote requirements.txt


In [21]:
# Cell 21b — .env.example
from pathlib import Path

Path(".env.example").parent.mkdir(parents=True, exist_ok=True)
Path(".env.example").write_text(
'# Copy this file to .env for local (non-Colab) experimentation only.\n# In Google Colab, DO NOT use a .env file — load the key from\n# Colab Secrets instead (see README.md, "Gemini API Setup").\n\nGEMINI_API_KEY=your-gemini-api-key-here\n\n# Optional overrides:\nGEMINI_MODEL=gemini-2.0-flash\nMAX_ITERATIONS=4\nEXECUTION_TIMEOUT=10\n',
    encoding="utf-8"
)
print("✓ wrote .env.example")

✓ wrote .env.example


In [22]:
# Cell 21c — data/sample_tasks.txt
from pathlib import Path

Path("data/sample_tasks.txt").parent.mkdir(parents=True, exist_ok=True)
Path("data/sample_tasks.txt").write_text(
'TASK 1 (Easy):\nCreate a Python function that calculates compound interest.\nWrite tests for positive and zero interest rates.\n\n---\n\nTASK 2 (Medium):\nCreate a CLI expense tracker.\n\nRequirements:\n- Add expense\n- Delete expense\n- List expenses\n- Calculate total\n- Save data to JSON\n\n---\n\nTASK 3 (Advanced):\nCreate a Python CSV data analyzer.\n\nRequirements:\n- Load a CSV file\n- Detect numeric columns\n- Calculate basic statistics\n- Handle missing values\n- Produce a readable summary\n\n---\n\nTASK 4 (Bonus):\nCreate a Python utility that validates and normalizes phone numbers\nfrom a list of raw strings, grouping valid and invalid entries.\n\n---\n\nTASK 5 (Bonus):\nCreate a simple command-line to-do list manager that stores tasks\nin a local JSON file and supports add, complete, and list operations.\n',
    encoding="utf-8"
)
print("✓ wrote data/sample_tasks.txt")

✓ wrote data/sample_tasks.txt


In [23]:
# Cell 21d — examples/expense_tracker_task.txt
from pathlib import Path

Path("examples/expense_tracker_task.txt").parent.mkdir(parents=True, exist_ok=True)
Path("examples/expense_tracker_task.txt").write_text(
'Build a Python CLI expense tracker.\n\nRequirements:\n- Add an expense\n- Delete an expense\n- List expenses\n- Show total spending\n- Store data locally\n',
    encoding="utf-8"
)
print("✓ wrote examples/expense_tracker_task.txt")

✓ wrote examples/expense_tracker_task.txt


In [24]:
# Cell 22 — README.md
from pathlib import Path

Path("README.md").parent.mkdir(parents=True, exist_ok=True)
Path("README.md").write_text(
'# 🤖 Agentic Coding Agent\n\n**Project 2 of 4 — Agentic AI Portfolio**\n\n```\n01 — Research Agent\n02 — Coding Agent        ← this project\n03 — Sales Agent\n04 — Recruiting Agent\n```\n\nBuilt and runnable **100% inside Google Colab**, using the **Gemini\nfree-tier API** — no OpenAI, no Docker, no LangChain, no paid services.\n\n---\n\n## Overview\n\nThis project is an autonomous coding agent. Given a natural-language\nprogramming task, it plans an implementation, writes the code, writes\ntests, runs everything locally, diagnoses failures, fixes the code,\nand re-tests — all inside a bounded loop with an explicit state\nobject — before reporting a final result.\n\n## Why This Is Agentic\n\n```\nLLM ≠ Agent\n```\n\nA single call to an LLM that returns code is not an agent — it\'s a\ntext completion. This project is agentic because it:\n\n```\nPlan → Act → Observe → Correct → Repeat\n```\n\nConcretely: the agent **plans** an approach, **acts** by generating\nand executing code, **observes** the real stdout/stderr/test results\nfrom that execution, **corrects** the code based on those\nobservations, and **repeats** until the tests pass or a hard\niteration limit is reached. The control flow, the state, and the\nstopping condition are all explicit Python — not hidden inside a\nprompt.\n\n## Architecture\n\n```\n                  USER TASK\n                      │\n                      ▼\n             ┌─────────────────┐\n             │ REQUIREMENT     │\n             │ ANALYZER        │\n             └────────┬────────┘\n                      ▼\n             ┌─────────────────┐\n             │    PLANNER      │\n             └────────┬────────┘\n                      ▼\n             ┌─────────────────┐\n             │ CODE GENERATOR  │\n             └────────┬────────┘\n                      ▼\n             ┌─────────────────┐\n             │ TEST GENERATOR  │\n             └────────┬────────┘\n                      ▼\n             ┌─────────────────┐\n             │ CODE EXECUTOR   │\n             └────────┬────────┘\n                      ▼\n                 TEST RESULT\n                 /        \\\n              PASS        FAIL\n               │            │\n               │            ▼\n               │    ┌───────────────┐\n               │    │ FAILURE       │\n               │    │ ANALYZER      │\n               │    └───────┬───────┘\n               │            ▼\n               │    ┌───────────────┐\n               │    │ CODE FIXER    │\n               │    └───────┬───────┘\n               │            │\n               │            └──────► RETEST\n               │\n               ▼\n        FINAL IMPLEMENTATION\n```\n\nCode layout mirrors this:\n\n| Concern                        | File               |\n|---------------------------------|--------------------|\n| Configuration / tunables        | `config.py`        |\n| Gemini client wrapper           | `llm.py`           |\n| Tools (generate/run/fix)        | `tools.py`         |\n| Agent state, actions, the loop  | `coding_agent.py`  |\n| Local, no-API unit tests        | `tests/test_tools.py` |\n\n## Agent Loop\n\n```python\nwhile not state.success and state.iteration < MAX_ITERATIONS:\n    action = decide_next_action(state)   # reasoning — no side effects\n    execute_action(action, state)        # tool execution — no reasoning\n```\n\n`_decide_next_action` is pure decision logic over the current\n`CodingState` — it never calls Gemini or runs code itself. Every\naction it can return maps to exactly one handler in\n`_execute_action`, which is the only place tools are invoked. This\nsplit is what keeps "what to do next" separate from "how it\'s done."\n\n`MAX_ITERATIONS = 4` is a hard ceiling — the loop can never spin\nforever, which matters a lot on a free-tier quota.\n\n## Tools\n\n| Tool               | Job                                                            |\n|---------------------|----------------------------------------------------------------|\n| `CodeGeneratorTool` | Task + requirements + plan → Python source                     |\n| `TestGeneratorTool` | Code → `unittest`-based tests                                  |\n| `CodeExecutor`      | Runs Python source in a temp dir with a timeout, captures I/O  |\n| `TestExecutor`      | Runs the tests the same way, parses pass/fail counts           |\n| `CodeFixerTool`     | Task + code + tests + failure output → diagnosis + fixed code  |\n\nEach tool does one job and knows nothing about the agent loop —\nthey\'re called, not consulted.\n\n## State Management\n\n```python\n@dataclass\nclass CodingState:\n    task: str\n    requirements: list\n    plan: list\n    code: str\n    tests: str\n    execution_results: list\n    errors: list\n    fixes: list\n    iteration: int\n    success: bool\n    final_output: str\n    gemini_calls_used: int\n```\n\nThe same `CodingState` instance is threaded through every iteration\nof the loop and mutated in place — this is what lets the agent\n"remember" the plan, the current code, and prior failures instead of\nstarting from scratch each time.\n\n## Code Generation\n\n`CodeGeneratorTool` asks Gemini for a single fenced ` ```python ` block\nimplementing the task per the plan, with type hints, docstrings, and\nbasic error handling. Output is parsed with `strip_markdown_fence`,\nwhich robustly handles fenced, unfenced, and mixed responses.\n\n## Testing\n\n`TestGeneratorTool` produces `unittest`-based tests (chosen over\npytest to avoid an extra dependency in Colab). `TestExecutor` runs\nthem with `python -m unittest ... -v` in an isolated temp directory\nand parses a `passed / failed / errors` summary from the output.\n\n## Self-Correction\n\nWhen tests fail, `CodeFixerTool.diagnose_and_fix` sends the task,\ncurrent code, tests, and captured failure output to Gemini, gets back\na short JSON diagnosis (`problem`, `cause`, `fix_strategy`), and then\nrequests a corrected full module. The fix becomes the new `state.code`\nand the loop re-runs the tests.\n\n## Security Considerations\n\n**This is an educational coding-agent prototype — not a hardened\nsandbox.** Generated code is executed locally with:\n\n- a fresh `tempfile.TemporaryDirectory()` per run\n- a `subprocess.run(..., timeout=EXECUTION_TIMEOUT, capture_output=True)`\n- no shell execution — only `python <file>` and `python -m unittest`\n  are ever invoked, never arbitrary LLM-generated shell commands\n- automatic cleanup when the temp dir context exits\n\n**Never run this against untrusted tasks or in a multi-tenant\nenvironment.** A production system would additionally need:\n\n- container isolation\n- non-root execution\n- CPU/memory limits\n- filesystem isolation\n- network restrictions\n- syscall restrictions (e.g. seccomp)\n- stronger sandboxing (e.g. gVisor, Firecracker)\n- audit logging\n\n## Project Structure\n\n```\n02-coding-agent/\n│\n├── README.md\n├── coding_agent.py\n├── tools.py\n├── llm.py\n├── config.py\n├── requirements.txt\n├── .env.example\n├── data/\n│   └── sample_tasks.txt\n├── examples/\n│   └── expense_tracker_task.txt\n└── tests/\n    └── test_tools.py\n\nnotebooks/\n└── Agentic_Coding_Agent_Colab.ipynb\n```\n\n## Tech Stack\n\n- Python 3 (standard library: `subprocess`, `tempfile`, `unittest`, `dataclasses`, `enum`, `json`)\n- `google-genai` (Gemini free-tier API)\n- Google Colab (Secrets, form fields, file generation, ZIP download)\n\nNo LangChain, LangGraph, CrewAI, AutoGen, Docker, Kubernetes,\ndatabases, Redis, cloud deployment, or frontend framework is used —\nthe agent architecture is implemented from first principles in plain\nPython so it\'s fully transparent.\n\n## Google Colab Setup\n\n1. Open `notebooks/Agentic_Coding_Agent_Colab.ipynb` in Google Colab.\n2. Run the cells top to bottom.\n3. The notebook generates every project file (`config.py`, `llm.py`,\n   `tools.py`, `coding_agent.py`, tests, README, etc.) directly onto\n   the Colab filesystem via `Path(...).write_text(...)` cells.\n\n## Gemini API Setup\n\n1. Get a free-tier API key from [Google AI Studio](https://aistudio.google.com/apikey).\n2. In Colab, click the 🔑 **Secrets** icon in the left sidebar.\n3. Add a new secret named `GEMINI_API_KEY` with your key as the value.\n4. Toggle **Notebook access** on for this secret.\n5. The notebook loads it with:\n   ```python\n   from google.colab import userdata\n   GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")\n   ```\n   The key is never printed or hard-coded anywhere in the project.\n\n## Running the Demo\n\nThe notebook includes three demo tasks (easy / medium / advanced) plus\na free-text cell for a fully custom task:\n\n```python\nagent = CodingAgent(api_key=GEMINI_API_KEY)\nstate = agent.run("Build a Python CLI expense tracker...")\nprint_final_report(state)\n```\n\n## Example Output\n\n```\n========================================================\n🤖 AGENTIC CODING AGENT\n========================================================\n\n🎯 Task:\nBuild a Python CLI expense tracker.\n\n🧠 Step — Understanding requirements & creating plan\n✓ 5 requirements identified\n✓ Plan created (5 steps)\n\n💻 Step — Generating code\n✓ Code generated\n\n🧪 Step — Generating tests\n✓ Tests generated\n\n▶ Step — Running tests\n✗ Tests failed (passed=3 failed=2 errors=0 (exit code 1))\n\n🔍 Step — Diagnosing failure\n✓ Root cause identified: JSON file not initialized before first read\n\n🔧 Step — Fixing code\n✓ Updated implementation\n\n▶ Step — Running tests\n✓ All tests passed (passed=5 failed=0 errors=0 (exit code 0))\n\n========================================================\n✅ CODING TASK COMPLETED\n========================================================\n```\n\n## Free-Tier Optimization\n\nGemini calls are deliberately batched to stay within a **3–5 live\ncalls per task** budget:\n\n| Call | Purpose                                     |\n|------|----------------------------------------------|\n| 1    | Requirements extraction + planning            |\n| 2    | Code generation                               |\n| 3    | Test generation                               |\n| 4–5  | Diagnose (1 call) + fix (1 call) — only if tests fail |\n\nTest execution, code execution, and re-testing all happen **locally**\nwith zero additional API calls. `MAX_ITERATIONS = 6` gives the agent\nexactly one diagnose-and-fix cycle when tests fail; `EXECUTION_TIMEOUT\n= 10` (seconds) bounds wall-clock time for running generated code.\n\nOn top of that call budget, `llm.py` layers three protections so a\nsmall free-tier key still produces a clean, complete notebook run:\n\n- **Disk cache** (`.gemini_cache/`) — identical `(model, prompt,\n  temperature, max_tokens)` requests are served from disk. Re-running\n  a cell while polishing this notebook costs zero quota.\n- **Pacing** (`MIN_SECONDS_BETWEEN_CALLS`, default 6s) — a minimum gap\n  is enforced between live calls so the notebook can\'t burst past a\n  requests-per-minute limit.\n- **Session budget** (`GEMINI_SESSION_CALL_BUDGET`, default 20) — a\n  hard cap on live calls per notebook run. Once reached, the agent\n  stops itself *gracefully* — printing partial results and a clear\n  "paused" status — instead of crashing on a raw `429` mid-demo.\n\nAll three are tunable via environment variables in `config.py` if your\nkey has a different (or more generous) quota.\n\n## Limitations\n\n- Python-only; no multi-language support in this version.\n- `subprocess` execution is a controlled demo, not a real sandbox.\n- Test-result parsing is simple string parsing over `unittest -v`\n  output, not a structured test report.\n- Only the most recent failure is used for diagnosis, not full history.\n- No persistence between runs — each `agent.run(task)` call starts a\n  fresh `CodingState`.\n\n## Future Improvements\n\n- Add a lightweight structured test report (e.g. `unittest`\'s\n  `TestResult` object) instead of parsing stderr text.\n- Support additional languages behind the same tool interface.\n- Add a proper sandbox (container or microVM) for real isolation.\n- Persist state across runs to resume interrupted tasks.\n- Add a retrieval step so the agent can reuse patterns from prior\n  successful tasks in this same project.\n\n## Author\n\nBuilt by Kunal as Project 2 of a 4-part Agentic AI portfolio,\ndemonstrating agent architecture, planning, tool use, code generation,\nautomated testing, error analysis, self-correction, and state\nmanagement — implemented from first principles in plain Python.\n',
    encoding="utf-8"
)
print("✓ wrote README.md")


✓ wrote README.md


In [25]:
# Verify every required file is present before packaging.
expected_files = [
    "README.md",
    "coding_agent.py",
    "tools.py",
    "llm.py",
    "config.py",
    "requirements.txt",
    ".env.example",
    "data/sample_tasks.txt",
    "examples/expense_tracker_task.txt",
    "tests/test_tools.py",
]

missing = [f for f in expected_files if not Path(f).exists()]
if missing:
    print("\u26a0 Missing files:", missing)
else:
    print("\u2713 All expected project files are present:")
    for f in expected_files:
        size = Path(f).stat().st_size
        print(f"    {f}  ({size} bytes)")

✓ All expected project files are present:
    README.md  (12973 bytes)
    coding_agent.py  (13573 bytes)
    tools.py  (13107 bytes)
    llm.py  (12401 bytes)
    config.py  (4691 bytes)
    requirements.txt  (20 bytes)
    .env.example  (326 bytes)
    data/sample_tasks.txt  (790 bytes)
    examples/expense_tracker_task.txt  (147 bytes)
    tests/test_tools.py  (6110 bytes)


In [26]:
# Package everything into a single GitHub-ready ZIP:
#   02-coding-agent/  (all project files, this ZIP's own contents)
#   notebooks/Agentic_Coding_Agent_Colab.ipynb  (this notebook, if present)

import zipfile

zip_name = "02-coding-agent.zip"
project_files = expected_files + (["final_solution.py"] if Path("final_solution.py").exists() else [])

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in project_files:
        zf.write(f, arcname=f"02-coding-agent/{f}")

    # Include this notebook in the ZIP too, if Colab has saved a copy
    # locally (e.g. via File > Save, or when running from a cloned repo).
    for candidate in Path(".").glob("*.ipynb"):
        zf.write(candidate, arcname=f"notebooks/{candidate.name}")

print(f"\u2713 Created {zip_name} ({Path(zip_name).stat().st_size} bytes)")

✓ Created 02-coding-agent.zip (23290 bytes)


In [27]:
# Download the ZIP (Colab only). Upload its contents directly to a
# GitHub repo as the `02-coding-agent/` folder of your Agentic AI
# portfolio.
try:
    from google.colab import files
    files.download(zip_name)
except Exception as e:
    print("Not running in Colab, or download was blocked:", e)
    print(f"You can find the file at: {Path(zip_name).resolve()}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>